In [ ]:
%pip uninstall torch torchvision torchaudio -y
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Restart the Notebook kernel after downloading to make sure the new libs are used.

In [1]:
# You may have PyTorch installed, but it may not be configured to use your GPU. Run the following code to check, if it returns "None", run the Install block.

import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce RTX 3050


## VJEPA-2 Testing and Running

- Below is where the processing pipeline starts, before each block there will be a markdown explaining their purpose and expected output.  
- The block right below this will install all the basic packages required for this Notebook.  
- The block above this will verify and provide you with options to install the correct dependency package for utilization of the GPU instead of CPU for training purposes.  

In [ ]:
# Python Version of this Notebook is 3.11.9

%pip install transformers
%pip install datasets
%pip install decord
%pip install numpy==1.26.4
%pip install Pillow==10.2.0
%pip install matplotlib==3.8.2

# Restart the Notebook kernel after downloading to make sure the new libs are used.

## 0. Imports and Environment Setup

In [ ]:
import torch
import torch.nn.functional as F

import numpy as np

from decord import VideoReader, cpu
from transformers import AutoVideoProcessor, AutoModel, AutoModelForVideoClassification
from PIL import Image

DEVICE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Tải phiên bản đã được huấn luyện sẵn trên Something-Something v2
model_id = "facebook/vjepa2-vitl-fpc16-256-ssv2" 
model = AutoModelForVideoClassification.from_pretrained(model_id).to(DEVICE)
processor = AutoVideoProcessor.from_pretrained(model_id)

# FIX: Mô hình fpc16 cần đúng 16 khung hình — lấy mẫu lại từ video_data (64 khung)
# indices_16 = np.linspace(0, len(video_data) - 1, 16, dtype=int)
# video_data_16 = video_data[indices_16]  # (16, H, W, 3)

inputs = processor(list(video_data), return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    
# Lấy nhãn hành động có xác suất cao nhất
predicted_class_idx = torch.argmax(logits, dim=-1).item()
print(f"Hành động dự đoán: {model.config.id2label[predicted_class_idx]}")


Loading weights: 100%|██████████| 652/652 [00:00<00:00, 3083.44it/s]


Hành động dự đoán: Scooping [something] up with [something]


In [ ]:
# Nạp bộ mã hóa và bộ dự đoán hành động (bản Giant 1.2B)
encoder, ac_predictor = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_ac_vit_giant')

encoder.to(DEVICE).eval()
ac_predictor.to(DEVICE).eval()

# 1. Trích xuất đặc trưng hình ảnh hiện tại (z_k)
# (Giả sử video_tensor đã được xử lý từ decord như ở ví dụ trước)
with torch.no_grad():
    # FIX: torch.hub encoder nhận đầu vào dạng (B, C, T, H, W) — float, chuẩn hoá về [0, 1]
    # video_tensor có dạng (T, C, H, W) → cần permute sang (C, T, H, W) rồi unsqueeze batch
    encoder_input = video_tensor.permute(1, 0, 2, 3).unsqueeze(0).float().div(255.0).to(DEVICE)
    z_k = encoder(encoder_input)

    # 4. Giả lập trạng thái robot (7 chiều: x, y, z, euler_x, euler_y, euler_z, gripper)
    # s_k: trạng thái hiện tại, a_k: hành động thực hiện
    s_k = torch.zeros(1, 32, 7).to(DEVICE)
    a_k = torch.zeros(1, 32, 7).to(DEVICE)

    # 5. Dự đoán trạng thái ẩn tiếp theo bằng ac_predictor
    # Predictor của bản Hub nhận vào 3 tham số trực tiếp: (embeddings, states, actions) 
    z_next_pred = ac_predictor(z_k, s_k, a_k)

print(f"Trích xuất thành công z_k: {z_k.shape}")
print(f"Dự đoán thành công tương lai z_next: {z_next_pred.shape}")

Using cache found in C:\Users\sangg/.cache\torch\hub\facebookresearch_vjepa2_main
c:\Users\sangg\AppData\Local\Programs\Python\Python314\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\sangg\AppData\Local\Programs\Python\Python314\Lib\contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Trích xuất thành công z_k: torch.Size([1, 8192, 1408])
Dự đoán thành công tương lai z_next: torch.Size([1, 8192, 1408])


In [ ]:


# 1. Nạp khung hình tiếp theo (next frame) để so sánh với embedding dự đoán
next_frame_path = "next_frame.jpg"
next_frame_img = Image.open(next_frame_path).convert("RGB")
next_frame_np = np.array(next_frame_img)

# 2. Chuyển đổi khung hình tiếp theo thành tensor và chuẩn hóa
next_frame_tensor = torch.from_numpy(next_frame_np).permute(2, 0, 1).float().div(255.0)
next_frame_tensor = next_frame_tensor.unsqueeze(1).repeat(1, 32, 1, 1)
next_frame_tensor = next_frame_tensor.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    z_true_next = encoder(next_frame_tensor)

# 3. Pooling (trung bình) qua chiều thời gian để có một vector embedding duy nhất cho khung hình tiếp theo
z_next_pred_pool = z_next_pred.mean(dim=1)
z_true_next_pool = z_true_next.mean(dim=1)

# 4. So sánh embedding dự đoán với embedding thực tế của khung hình tiếp theo
cos_sim = F.cosine_similarity(z_next_pred_pool, z_true_next_pool, dim=-1)
mse = F.mse_loss(z_next_pred_pool, z_true_next_pool)
l2 = torch.norm(z_next_pred_pool - z_true_next_pool, dim=-1)

print(f"z_next_pred shape: {z_next_pred.shape}")
print(f"z_true_next shape: {z_true_next.shape}")
print(f"Cosine similarity: {cos_sim.item():.4f}")
print(f"MSE: {mse.item():.6f}")
print(f"L2 distance: {l2.item():.6f}")

z_next_pred shape: torch.Size([1, 8192, 1408])
z_true_next shape: torch.Size([1, 4096, 1408])
Cosine similarity: 0.7904
MSE: 1.173576
L2 distance: 40.649666
